# Смена директории


In [1]:
%cd ..

/Users/uzumnasiya/HSE/Year_Project


# Импорт библиотек


In [2]:
import logging
import warnings
import time


import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import yaml
import joblib
import shap
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from category_encoders import CatBoostEncoder
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


from utils.dev_utils import get_pool
from utils.pipeline_utils import CustomPreprocessor
from utils.metrics import MetricCalculator, metric_funcs
from utils.style.styler import style_metrics
from utils.style.html_output import print_multiple_html
from utils.plot_utils import plot_gini_by_period_styled, plot_roc_by_masks
import os
import mlflow
from dotenv import load_dotenv


In [3]:
logging.getLogger().setLevel(logging.WARNING)
warnings.filterwarnings('ignore')
sns.set_palette('bright')


pd.options.display.float_format = "{:.2f}".format
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

# Загрузка PRD-модели из MLflow


In [4]:
load_dotenv()
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'admin')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'password')
s3 = os.getenv('MLFLOW_S3_ENDPOINT_URL', 'http://localhost:9000')
if 'minio:9000' in s3:
    s3 = 'http://localhost:9000'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = s3
mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5050'))

REGISTERED_MODEL_NAME = 'year_project_fraud_catboost'
loaded = mlflow.pyfunc.load_model(f'models:/{REGISTERED_MODEL_NAME}@prd')


## Тестовый predict


In [10]:
with open(r'./models/params/features.yaml', encoding='utf-8') as file:
    features_cfg = yaml.safe_load(file)

FINAL_FEATURES = features_cfg['FINAL_FEATURES']
FINAL_CAT_FEATURES = features_cfg['FINAL_CAT_FEATURES']

data = pd.read_parquet('./data/processed/data.pqt')
sample = CustomPreprocessor(cat_features=FINAL_CAT_FEATURES).transform(
    data.loc[data['sample_type'] == 'TEST'].head(15)
)

X = sample[FINAL_FEATURES].copy()
for col in FINAL_CAT_FEATURES:
    X[col] = X[col].astype(str)

cb_model = mlflow.catboost.load_model(f"models:/{REGISTERED_MODEL_NAME}@prd")
proba = cb_model.predict_proba(X)[:, 1]

pd.DataFrame({'target': sample['target'].values, 'score': proba})


,target,score
0,0.00,0.00
1,0.00,0.00
2,0.00,0.00
3,0.00,0.01
4,0.00,0.00
5,0.00,0.01
6,0.00,0.00
7,0.00,0.00
8,0.00,0.00
9,0.00,0.00
